In [25]:
import os
import glob
import pandas as pd

# Define the target directory
data_dir = "../data_curation/data"

# Find all Parquet files, excluding 'games.parquet' and anything with 'archive'
parquet_files = [
    f for f in glob.glob(os.path.join(data_dir, "*.parquet")) 
]

if not parquet_files:
    print(f"No matching .parquet files found in {data_dir}")
else:
    for file_path in parquet_files:
        file_name = os.path.basename(file_path)
        
        print("=" * 60)
        print(f"FILE: {file_name}")
        print("=" * 60)
        
        try:
            # Read only the schema/metadata to get columns instantly without loading the whole file
            # This is significantly faster for large Parquet files
            columns = pd.read_parquet(file_path, engine='pyarrow').columns.tolist()
            
            print(f"Total Columns: {len(columns)}")
            print("-" * 40)
            print("Columns list (Ready to copy):")
            print(columns)
            
            # Print as a clean vertical list as well for easy scanning
            # print("\n".join(columns))
                
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            
        print("\n")

FILE: GameSummaries.parquet
Total Columns: 8
----------------------------------------
Columns list (Ready to copy):
['game_id', 'game_code', 'game_status', 'game_time_utc', 'arena_name', 'arena_city', 'attendance', 'sellout_flag']


FILE: BoxScoresSummary.parquet
Total Columns: 28
----------------------------------------
Columns list (Ready to copy):
['game_id', 'game_code', 'game_status', 'game_status_text', 'period', 'game_time_utc', 'game_et', 'game_duration', 'attendance', 'sellout', 'home_team_id', 'home_team_tricode', 'home_team_name', 'home_team_city', 'home_q1', 'home_q2', 'home_q3', 'home_q4', 'home_score', 'away_team_id', 'away_team_tricode', 'away_team_name', 'away_team_city', 'away_q1', 'away_q2', 'away_q3', 'away_q4', 'away_score']


FILE: PlayerBoxScores.parquet
Total Columns: 6
----------------------------------------
Columns list (Ready to copy):
['game_id', 'team_id', 'player_id', 'player_name', 'roster_status', 'dnp_comment']


FILE: AdvBoxScoresMiscPlayoffs.parquet
T

In [12]:
import random
from pathlib import Path
from bs4 import BeautifulSoup

# Define the path to the directory containing the HTML files
directory_path = Path("../data_curation/data/unscraped_sites/usatoday_sag/html/")

try:
    if not directory_path.exists() or not directory_path.is_dir():
        print(f"Error: The path {directory_path.resolve()} is not a valid directory.")
    else:
        # Collect all HTML files in the folder
        html_files = list(directory_path.glob("*.html"))
        total_files = len(html_files)
        
        if total_files == 0:
            print(f"No HTML files found in {directory_path.resolve()}")
        else:
            # Determine sample size (use 10, or less if the directory has fewer than 10 files)
            sample_size = min(10, total_files)
            sampled_files = random.sample(html_files, sample_size)
            
            print(f"--- Sampling {sample_size} random files out of {total_files} total files ---")
            print("=" * 80)
            
            for file_index, file_path in enumerate(sampled_files, start=1):
                print(f"\nPROCESSING FILE {file_index}/{sample_size}: file:///Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/unscraped_sites/usatoday_sag/html/{file_path.name}")
                print("=" * 80)
                
                try:
                    # Read and parse the content of the file
                    html_content = file_path.read_text(encoding="utf-8")
                    soup = BeautifulSoup(html_content, "html.parser")
                    
                    # Select the parent element
                    sagarin_element = soup.select_one("#sagarin")
                    
                    if sagarin_element:
                        print("--- RAW HTML STRUCTURE OF #sagarin ---")
                        print(sagarin_element.prettify())
                        
                        print("\n--- IMMEDIATE CHILD ELEMENTS ---")
                        # Extract and count only the actual HTML tags directly inside #sagarin
                        children = [child for child in sagarin_element.children if child.name is not None]
                        print(f"Found {len(children)} immediate sub-elements inside #sagarin.\n")
                        
                        for i, child in enumerate(children, start=1):
                            print(f"Child {i}: <{child.name}> (Class: {child.get('class', 'None')})")
                            # Print a snippet of this specific sub-element's text
                            child_text = child.get_text(strip=True)
                            snippet = child_text[:60] + "..." if len(child_text) > 60 else child_text
                            print(f"  Snippet: {snippet}\n")
                            
                    else:
                        print(f"Warning: Selector '#sagarin' was not found in {file_path.name}")
                        
                except Exception as file_error:
                    print(f"An error occurred while processing file {file_path.name}: {file_error}")
                
                print("-" * 80)

except Exception as e:
    print(f"An unexpected error occurred during directory scanning: {e}")

--- Sampling 10 random files out of 2237 total files ---

PROCESSING FILE 1/10: file:///Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/unscraped_sites/usatoday_sag/html/20130620174145.html
--- RAW HTML STRUCTURE OF #sagarin ---
<article data-page="sagarin" id="sagarin">
 <div class="sagarin-page">
  <font>
   <pre></pre>
  </font>
  <pre><br/>All Contents Copyright © 2013 Jeff Sagarin<sup><font size="1">tm</font></sup>. All rights reserved.<br/><b><br/>NBA 2012-2013 ratings through results of 2013 JUNE 18 TUESDAY - NBA Finals<br/></b><br/><br/>this output has three parts: (1) teams listed by RATING top-to-bottom<br/>                             (2) DIVISION AVERAGES<br/>                                 (listed top-to-bottom &amp; by conference)<br/>                             (3) teams listed by DIVISION<br/>                                 (listed in order within divisions)<br/><br/>The SCHEDULE ratings represent what the rating would have to be for a<br/>hypo


## Is there a way we can sanity check this or verify that it is parsing correctly without me random sammpling some files and comparing them to the data we parsed?


Value bounds
- Ratings should be between ~75 and ~110. Flag anything outside that
- wins + losses should be reasonable for the point in the season (e.g. can't be > 82 + playoff games)
- home_advantage should be between 1.5 and 5.0 — flag outliers

Cross-column consistency
- For 3col rows: predictor, golden_mean, recent should all be within ~10 points of rating (they're components of it)
- predictor_rank for the team with the highest predictor value should be 1

Temporal consistency
- Ratings for a given team shouldn't jump more than ~10 points between consecutive snapshots — large jumps indicate a parsing error or misidentified team
- Team names should be stable — a sudden new team name appearing mid-season likely means a regex captured something wrong

Known ground truth spot checks (hardcoded)
- Golden State Warriors 2015-16 final: rating ≈ 99.13 — you can assert this from a known-good snapshot date
- A handful of these pinned assertions will give you high confidence the group mappings are correct across format variants

The rank monotonicity + 30-team count check is the fastest signal — if every snapshot has exactly 30 uniquely-ranked teams, the parser is almost certainly working correctly.

In [50]:
import pandas as pd
df = pd.read_parquet('../data_curation/data/sag_ratings_parsed.parquet')
missing_pct = df['as_of_date'].isna().mean() * 100
print(f"Percentage of rows with missing dates: {missing_pct:.2f}%")

# To see the exact count breakdown of missing vs valid rows
df['as_of_date'].isna().value_counts()

missing = df[df['as_of_date'].isna()]
missing

Percentage of rows with missing dates: 0.85%


,sag_rank,team,sag_rating,wins,losses,schedl,schedl_rank,elo_score,elo_score_rank,predictor,...,home_advantage,source_file,pure_elo,pure_elo_rank,dimin_curve,dimin_curve_rank,golden_mean,golden_mean_rank,recent,recent_rank
66510,1,San Antonio Spurs,96.62,0,0,0.0,0,96.62,1.0,96.62,...,2.71,20141028093141.html,NaN,NaN,NaN,NaN,96.62,1.0,NaN,NaN
66511,1,San Antonio Spurs,96.62,0,0,0.0,0,96.62,1.0,96.62,...,2.71,20141027042237.html,NaN,NaN,NaN,NaN,96.62,1.0,NaN,NaN
66512,1,Golden State Warriors,96.99,0,0,0.0,0,NaN,NaN,96.99,...,2.44,20151026203840.html,NaN,NaN,NaN,NaN,96.99,1.0,96.99,1.0
66513,1,Golden State Warriors,96.99,0,0,0.0,0,NaN,NaN,96.99,...,2.44,20151028014224.html,NaN,NaN,NaN,NaN,96.99,1.0,96.99,1.0
66514,1,Golden State Warriors,101.94,0,0,0.0,0,NaN,NaN,101.94,...,3.07,20171017095829.html,NaN,NaN,NaN,NaN,101.94,1.0,101.94,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67075,30,Charlotte Hornets,82.16,0,0,0.0,0,NaN,NaN,82.16,...,2.71,20191022105318.html,NaN,NaN,NaN,NaN,82.16,30.0,82.16,30.0
67076,30,Charlotte Hornets,82.16,0,0,0.0,0,NaN,NaN,82.16,...,2.71,20191023122612.html,NaN,NaN,NaN,NaN,82.16,30.0,82.16,30.0
67077,30,Oklahoma City Thunder,83.41,0,0,0.0,0,NaN,NaN,83.41,...,2.11,20201222004432.html,NaN,NaN,NaN,NaN,83.41,30.0,83.41,30.0
67078,30,Oklahoma City Thunder,83.41,0,0,0.0,0,NaN,NaN,83.41,...,2.11,20201223022408.html,NaN,NaN,NaN,NaN,83.41,30.0,83.41,30.0


In [48]:
import pandas as pd
import numpy as np

def run_master_sagarin_validation(file_path: str) -> bool:
    """
    Executes structural and mathematical validation checks specifically targeting
    the master anchor columns: sag_rank and sag_rating.
    
    Parameters:
    -----------
    file_path : str
        Path to the parquet file containing the parsed Sagarin data.
        
    Returns:
    --------
    bool
        True if all critical master checks pass, False if anomalies are detected.
    """
    print("=" * 75)
    print(f"STARTING MASTER COLUMN VALIDATION SUITE FOR: {file_path}")
    print("=" * 75)
    
    # 1. Load Dataset
    try:
        df = pd.read_parquet(file_path)
        print(f"[INFO] Successfully loaded {len(df)} records.")
    except Exception as e:
        print(f"[CRITICAL ERROR] Failed to load data. Details: {e}")
        return False

    # Standardize types for the master metrics to prevent casting or NaN alignment mismatches
    df['as_of_date'] = pd.to_datetime(df['as_of_date'])
    df['sag_rating'] = pd.to_numeric(df['sag_rating'], errors='coerce')
    df['sag_rank'] = pd.to_numeric(df['sag_rank'], errors='coerce')
    df['wins'] = pd.to_numeric(df['wins'], errors='coerce')
    df['losses'] = pd.to_numeric(df['losses'], errors='coerce')
    
    validation_status = True
    
    # -------------------------------------------------------------------------
    # Check 1: Master Rank Bounds (1 to 33)
    # -------------------------------------------------------------------------
    out_of_bounds_rank = df[(df['sag_rank'] < 1) | (df['sag_rank'] > 33)]
    if not out_of_bounds_rank.empty:
        print(f"[FAIL] Check 1: Found {len(out_of_bounds_rank)} records with sag_rank outside [1, 33].")
        print(out_of_bounds_rank[['source_file', 'team', 'sag_rank']].head(10))
        validation_status = False
    else:
        print("[PASS] Check 1: All sag_rank values fall within the acceptable [1, 33] range.")

    # -------------------------------------------------------------------------
    # Check 2: Wins and Losses Integrity
    # -------------------------------------------------------------------------
    negative_records = df[(df['wins'] < 0) | (df['losses'] < 0)]
    if not negative_records.empty:
        print(f"[FAIL] Check 2a: Found {len(negative_records)} records with negative win/loss counts.")
        validation_status = False
    else:
        print("[PASS] Check 2a: No negative win or loss counts detected.")

    df['total_games'] = df['wins'] + df['losses']
    excessive_games = df[df['total_games'] > 82]
    if not excessive_games.empty:
        distinct_playoff_files = excessive_games['source_file'].unique()
        print(f"[WARN] Check 2b: Found {len(excessive_games)} rows across {len(distinct_playoff_files)} snapshots exceeding 82 games (Potential Postseason).")
    else:
        print("[PASS] Check 2b: All snapshots strictly observe the standard ≤ 82 regular-season game cap.")

    # -------------------------------------------------------------------------
    # Check 3: Master Monotonicity and 30-Team Structural Count Check
    # -------------------------------------------------------------------------
    snapshot_groups = df.groupby('source_file')
    invalid_team_counts = 0
    invalid_rank_counts = 0
    
    for file_name, group in snapshot_groups:
        unique_teams_count = group['team'].nunique()
        unique_ranks_count = group['sag_rank'].dropna().nunique()
        
        if unique_teams_count != 30:
            invalid_team_counts += 1
        if unique_ranks_count != 30:
            invalid_rank_counts += 1

    if invalid_team_counts > 0 or invalid_rank_counts > 0:
        print(f"[FAIL] Check 3: Snapshot uniformity rules broken for master columns.")
        print(f"       Snapshots lacking exactly 30 unique teams: {invalid_team_counts}")
        print(f"       Snapshots lacking exactly 30 unique sag_rank entries: {invalid_rank_counts}")
        validation_status = False
    else:
        print("[PASS] Check 3: Monotonicity verified. Every snapshot contains exactly 30 uniquely-ranked teams.")

    # -------------------------------------------------------------------------
    # Check 4: Rating Stability (Max 10 point jump across small date gaps)
    # -------------------------------------------------------------------------
    df_sorted = df.sort_values(by=['team', 'as_of_date'])
    df_sorted['rating_diff'] = df_sorted.groupby('team')['sag_rating'].diff().abs()
    df_sorted['days_diff'] = df_sorted.groupby('team')['as_of_date'].diff().dt.days
    
    unstable_ratings = df_sorted[(df_sorted['days_diff'] <= 7) & (df_sorted['rating_diff'] > 10)]
    if not unstable_ratings.empty:
        print(f"[FAIL] Check 4: Detected {len(unstable_ratings)} instances of anomalous sag_rating jumps (> 10 points within 7 days).")
        print(unstable_ratings[['as_of_date', 'team', 'sag_rating', 'rating_diff', 'days_diff']].head(10))
        validation_status = False
    else:
        print("[PASS] Check 4: Team sag_rating metrics demonstrate temporal stability across small date intervals.")

    # -------------------------------------------------------------------------
    # Check 5: Max Sagarin Rating Must Equal Rank 1
    # -------------------------------------------------------------------------
    rank_alignment_failures = 0
    for file_name, group in snapshot_groups:
        clean_group = group.dropna(subset=['sag_rating', 'sag_rank'])
        if not clean_group.empty:
            max_rating_idx = clean_group['sag_rating'].idxmax()
            top_team_row = clean_group.loc[max_rating_idx]
            if top_team_row['sag_rank'] != 1:
                rank_alignment_failures += 1

    if rank_alignment_failures > 0:
        print(f"[FAIL] Check 5: Highest sag_rating value does not resolve to sag_rank 1 in {rank_alignment_failures} snapshots.")
        validation_status = False
    else:
        print("[PASS] Check 5: Highest sag_rating value aligns perfectly with sag_rank 1 across all historical snapshots.")

    # -------------------------------------------------------------------------
    # Check 6: Equal Team Appearance Frequencies (Modern Era Only: 2015+)
    # -------------------------------------------------------------------------
    # Filter for the modern era where team names are stable and finalized
    df_modern = df[df['as_of_date'].dt.year >= 2015]
    
    if df_modern.empty:
        print("[WARN] Check 6: Skinned. No data records found for years >= 2015.")
    else:
        team_counts_modern = df_modern['team'].value_counts()
        if team_counts_modern.nunique() != 1:
            print(f"[FAIL] Check 6: Dataset imbalance detected in the modern era (2015+).")
            print("       Teams do not appear an equal number of times within stable naming windows:")
            print(team_counts_modern)
            validation_status = False
        else:
            print(f"[PASS] Check 6: Index balance verified for modern era (2015+). All 30 stable teams appear exactly {team_counts_modern.iloc[0]} times.")

    print("=" * 75)
    if validation_status:
        print("FINAL STATUS: SUCCESS - Master anchors (sag_rank, sag_rating) pass all validations.")
    else:
        print("FINAL STATUS: FAILED - Structural parsing anomalies detected on master anchors.")
    print("=" * 75)
    
    return validation_status

# Execution
if __name__ == "__main__":
    DATA_PATH = '../data_curation/data/sag_ratings_parsed.parquet'
    run_master_sagarin_validation(DATA_PATH)

STARTING MASTER COLUMN VALIDATION SUITE FOR: ../data_curation/data/sag_ratings_parsed.parquet
[INFO] Successfully loaded 67080 records.
[PASS] Check 1: All sag_rank values fall within the acceptable [1, 33] range.
[PASS] Check 2a: No negative win or loss counts detected.
[WARN] Check 2b: Found 14989 rows across 988 snapshots exceeding 82 games (Potential Postseason).
[PASS] Check 3: Monotonicity verified. Every snapshot contains exactly 30 uniquely-ranked teams.
[PASS] Check 4: Team sag_rating metrics demonstrate temporal stability across small date intervals.
[PASS] Check 5: Highest sag_rating value aligns perfectly with sag_rank 1 across all historical snapshots.
[PASS] Check 6: Index balance verified for modern era (2015+). All 30 stable teams appear exactly 1720 times.
FINAL STATUS: SUCCESS - Master anchors (sag_rank, sag_rating) pass all validations.


In [49]:
import pandas as pd
import numpy as np
import os

# 1. Load the dataset
df = pd.read_parquet('../data_curation/data/sag_ratings_parsed.parquet')
df['as_of_date'] = pd.to_datetime(df['as_of_date'], errors='coerce')
df['wins'] = pd.to_numeric(df['wins'], errors='coerce')
df['losses'] = pd.to_numeric(df['losses'], errors='coerce')
df['total_games'] = df['wins'] + df['losses']

RAW_HTML_DIR = "../data_curation/data/unscraped_sites/usatoday_sag/html/"

print("=" * 85)
print("RUNNING TARGETED DATE SANITY CHECKS")
print("=" * 85)

# -------------------------------------------------------------------------
# CATEGORY A: Target the 0.85% Missing Dates (NaT)
# -------------------------------------------------------------------------
nat_files = df[df['as_of_date'].isna()]['source_file'].unique()
print(f"[!] Found {len(nat_files)} unique source files generating NaT dates.")

# -------------------------------------------------------------------------
# CATEGORY B: Check for Mid-Season Game Count Regressions (Time-Travel)
# -------------------------------------------------------------------------
# We extract year from date to roughly group by season
df_clean = df.dropna(subset=['as_of_date', 'total_games']).copy()
df_clean['year'] = df_clean['as_of_date'].dt.year
df_clean = df_clean.sort_values(by=['team', 'as_of_date'])

# Calculate the difference in games played between consecutive snapshots for the same team
df_clean['game_diff'] = df_clean.groupby(['team', 'year'])['total_games'].diff()

# A negative value means games played DROPPED as time went forward (Chronological error)
time_travel_records = df_clean[df_clean['game_diff'] < 0]
time_travel_files = time_travel_records['source_file'].unique()
print(f"[!] Found {len(time_travel_files)} snapshots where game volumes regress chronologically.")

# -------------------------------------------------------------------------
# COMPILING THE SMART AUDIT LIST
# -------------------------------------------------------------------------
# We strategically combine the problematic files to review them
smart_sample = []

# Add up to 4 files that have NaT errors
if len(nat_files) > 0:
    smart_sample.extend(pd.Series(nat_files).sample(n=min(4, len(nat_files)), random_state=42).tolist())

# Add up to 4 files that caused game count regressions
if len(time_travel_files) > 0:
    smart_sample.extend(pd.Series(time_travel_files).sample(n=min(4, len(time_travel_files)), random_state=42).tolist())

# Fill up the rest with normal files to act as control baselines
all_unique_files = df['source_file'].unique()
remaining_slots = 10 - len(smart_sample)
if remaining_slots > 0:
    control_files = pd.Series([f for f in all_unique_files if f not in smart_sample])
    if not control_files.empty:
        smart_sample.extend(control_files.sample(n=min(remaining_slots, len(control_files)), random_state=42).tolist())

# -------------------------------------------------------------------------
# RENDER AUDIT LOGS FOR THE 10 TARGETED FILES
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print("DISPLAYING TARGETED AUDIT SELECTION (ANOMALIES + CONTROLS)")
print("=" * 85)

for idx, file_name in enumerate(smart_sample, 1):
    file_data = df[df['source_file'] == file_name].sort_values(by='sag_rank')
    parsed_date = file_data['as_of_date'].iloc[0] if not file_data.empty else "UNKNOWN"
    
    # Label the file so you know why it was picked
    label = "CONTROL"
    if file_name in nat_files:
        label = "ANOMALY: MISSING DATE (NaT)"
    elif file_name in time_travel_files:
        label = "ANOMALY: GAME COUNT REGRESSION"
        
    print(f"\n[{idx}] SOURCE FILE: {file_name} | [{label}]")
    print(f"    -> Parsed Timeline Date: {parsed_date}")
    print("-" * 75)
    
    # Display snapshot sample
    display(pd.concat([file_data.head(2), file_data.tail(2)])[[
        'sag_rank', 'team', 'sag_rating', 'wins', 'losses', 'total_games', 'as_of_date'
    ]])
    
    full_html_path = os.path.join(RAW_HTML_DIR, file_name)
    print(f"👉 Open in VS Code:  code {full_html_path}")
    print(f"👉 Open in Safari:   open -a Safari {full_html_path}")

RUNNING TARGETED DATE SANITY CHECKS
[!] Found 19 unique source files generating NaT dates.
[!] Found 9 snapshots where game volumes regress chronologically.

DISPLAYING TARGETED AUDIT SELECTION (ANOMALIES + CONTROLS)

[1] SOURCE FILE: 20141028093141.html | [ANOMALY: MISSING DATE (NaT)]
    -> Parsed Timeline Date: NaT
---------------------------------------------------------------------------


,sag_rank,team,sag_rating,wins,losses,total_games,as_of_date
66510,1,San Antonio Spurs,96.62,0,0,0,NaT
66529,2,Cleveland Cavaliers,96.40,0,0,0,NaT
67042,29,Milwaukee Bucks,82.80,0,0,0,NaT
67061,30,Philadelphia 76ers,78.59,0,0,0,NaT


👉 Open in VS Code:  code ../data_curation/data/unscraped_sites/usatoday_sag/html/20141028093141.html
👉 Open in Safari:   open -a Safari ../data_curation/data/unscraped_sites/usatoday_sag/html/20141028093141.html

[2] SOURCE FILE: 20171018115446.html | [ANOMALY: MISSING DATE (NaT)]
    -> Parsed Timeline Date: NaT
---------------------------------------------------------------------------


,sag_rank,team,sag_rating,wins,losses,total_games,as_of_date
66515,1,Golden State Warriors,101.94,0,0,0,NaT
66534,2,Houston Rockets,95.99,0,0,0,NaT
67047,29,Atlanta Hawks,83.04,0,0,0,NaT
67066,30,Chicago Bulls,81.92,0,0,0,NaT


👉 Open in VS Code:  code ../data_curation/data/unscraped_sites/usatoday_sag/html/20171018115446.html
👉 Open in Safari:   open -a Safari ../data_curation/data/unscraped_sites/usatoday_sag/html/20171018115446.html

[3] SOURCE FILE: 20191020083704.html | [ANOMALY: MISSING DATE (NaT)]
    -> Parsed Timeline Date: NaT
---------------------------------------------------------------------------


...

👉 Open in VS Code:  code ../data_curation/data/unscraped_sites/usatoday_sag/html/20131101212003.html
👉 Open in Safari:   open -a Safari ../data_curation/data/unscraped_sites/usatoday_sag/html/20131101212003.html

[9] SOURCE FILE: 20130604204937.html | [CONTROL]
    -> Parsed Timeline Date: 2013-06-03 00:00:00
---------------------------------------------------------------------------


,sag_rank,team,sag_rating,wins,losses,total_games,as_of_date
1680,1,San Antonio Spurs,98.04,70,26,96,2013-06-03
1683,2,Miami Heat,97.96,78,20,98,2013-06-03
1764,29,Orlando Magic,82.89,20,62,82,2013-06-03
1767,30,Charlotte Bobcats,81.15,21,61,82,2013-06-03


👉 Open in VS Code:  code ../data_curation/data/unscraped_sites/usatoday_sag/html/20130604204937.html
👉 Open in Safari:   open -a Safari ../data_curation/data/unscraped_sites/usatoday_sag/html/20130604204937.html

[10] SOURCE FILE: 20150101071531.html | [CONTROL]
    -> Parsed Timeline Date: 2014-12-31 00:00:00
---------------------------------------------------------------------------


,sag_rank,team,sag_rating,wins,losses,total_games,as_of_date
14880,1,Golden State Warriors,98.33,25,5,30,2014-12-31
14881,2,Portland Trail Blazers,96.48,26,7,33,2014-12-31
14908,29,Minnesota Timberwolves,81.60,5,25,30,2014-12-31
14909,30,Philadelphia 76ers,79.57,4,26,30,2014-12-31


👉 Open in VS Code:  code ../data_curation/data/unscraped_sites/usatoday_sag/html/20150101071531.html
👉 Open in Safari:   open -a Safari ../data_curation/data/unscraped_sites/usatoday_sag/html/20150101071531.html


In [52]:
import os
import glob
import pandas as pd

# Define the target directory relative to the notebook location
TARGET_DIR = "../data_curation/data"

def audit_parquet_files(directory_path):
    """
    Scans the target directory for Parquet files and generates a comprehensive
    audit report including schemas, data types, sample rows, and NA diagnostics.
    """
    # Find all .parquet or .pq files in the directory
    search_path = os.path.join(directory_path, "**", "*.parquet")
    parquet_files = glob.glob(search_path, recursive=True)
    
    # Also catch alternative extension .pq
    search_path_pq = os.path.join(directory_path, "**", "*.pq")
    parquet_files.extend(glob.glob(search_path_pq, recursive=True))

    parquet_files = [f for f in parquet_files if "Adv" not in f]
    
    if not parquet_files:
        print(f"No Parquet files found in target directory: {directory_path}")
        return

    print(f"Found {len(parquet_files)} Parquet file(s) for auditing.\n")
    print("=" * 80)

    for file_path in parquet_files:
        print(f"\nAUDIT REPORT FOR: {file_path}")
        print("=" * 80)
        
        try:
            # Read the parquet file into a pandas DataFrame
            df = pd.read_parquet(file_path)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
            print("-" * 80)
            continue
            
        total_rows = len(df)
        total_cols = len(df.columns)
        print(f"Dimensions: {total_rows} rows | {total_cols} columns")
        
        # 1. Columns and Data Types
        print("\n### 1. Columns & Data Types")
        dtype_df = pd.DataFrame({
            'Column': df.columns,
            'Data Type': df.dtypes.astype(str)
        }).reset_index(drop=True)
        display(dtype_df)
        
        # 2. Example Rows (Head and Tail)
        print("\n### 2. Example Rows (Head & Tail Snippet)")
        if total_rows <= 5:
            display(df)
        else:
            # Show top 3 and bottom 2 rows to give a holistic view of the file structure
            display(pd.concat([df.head(3), df.tail(2)]))
            
        # 3. NA Invalidation Audit
        print("\n### 3. Missing Value (NA) Analysis")
        na_counts = df.isna().sum()
        columns_with_na = na_counts[na_counts > 0]
        
        if columns_with_na.empty:
            print("Clean Sheet: Zero NA values detected across all columns.")
        else:
            # Calculate metrics for columns containing NA values
            na_report = pd.DataFrame({
                'Column with NA': columns_with_na.index,
                'Missing Count': columns_with_na.values,
                '% of Total Rows': ((columns_with_na.values / total_rows) * 100).round(2)
            }).reset_index(drop=True)
            
            display(na_report)
            
            # Extract and display sample rows where at least one NA value is present
            print("\n### 4. Sample Rows Containing NA Values")
            na_rows_mask = df.isna().any(axis=1)
            total_na_rows = na_rows_mask.sum()
            print(f"Total rows containing at least one NA: {total_na_rows} ({((total_na_rows / total_rows) * 100).round(2)}%)")
            
            # Display up to 5 examples of rows with missing data
            display(df[na_rows_mask].head(5))
            
        print("\n" + "-" * 80)

# Run the audit execution
audit_parquet_files(TARGET_DIR)

Found 10 Parquet file(s) for auditing.


AUDIT REPORT FOR: ../data_curation/data/GameSummaries.parquet
Dimensions: 39983 rows | 8 columns

### 1. Columns & Data Types


,Column,Data Type
0,game_id,str
1,game_code,str
2,game_status,int64
3,game_time_utc,str
4,arena_name,str
5,arena_city,str
6,attendance,int64
7,sellout_flag,int64



### 2. Example Rows (Head & Tail Snippet)


,game_id,game_code,game_status,game_time_utc,arena_name,arena_city,attendance,sellout_flag
0,0020600867,20070301/LACSEA,3,2007-03-02T03:30:00Z,KeyArena,Seattle,14601,0
1,0021601026,20170317/ORLPHX,3,2017-03-18T02:00:00Z,Talking Stick Resort Arena,Phoenix,16880,0
2,0029600833,19970302/UTAVAN,3,1997-03-02T20:00:00Z,GM Place,Vancouver,16415,0
39981,0022500096,20251024/WASDAL,3,2025-10-25T00:30:00Z,American Airlines Center,Dallas,19225,1
39982,0020401152,20050411/GSWDEN,3,2005-04-12T01:00:00Z,Pepsi Center,Denver,18045,0



### 3. Missing Value (NA) Analysis
Clean Sheet: Zero NA values detected across all columns.

--------------------------------------------------------------------------------

AUDIT REPORT FOR: ../data_curation/data/BoxScoresSummary.parquet
Dimensions: 84 rows | 28 columns

### 1. Columns & Data Types


...


### 3. Missing Value (NA) Analysis


,Column with NA,Missing Count,% of Total Rows
0,elo_score,59370,88.51
1,elo_score_rank,59370,88.51
2,as_of_date,570,0.85
3,pure_elo,57750,86.09
4,pure_elo_rank,57750,86.09
5,dimin_curve,66600,99.28
6,dimin_curve_rank,66600,99.28
7,golden_mean,4740,7.07
8,golden_mean_rank,4740,7.07
9,recent,17040,25.40



### 4. Sample Rows Containing NA Values
Total rows containing at least one NA: 67080 (100.0%)


,sag_rank,team,sag_rating,wins,losses,schedl,schedl_rank,elo_score,elo_score_rank,predictor,...,home_advantage,source_file,pure_elo,pure_elo_rank,dimin_curve,dimin_curve_rank,golden_mean,golden_mean_rank,recent,recent_rank
0,1,Oklahoma City Thunder,99.51,58,21,90.06,14,99.07,2.0,100.05,...,3.16,20130413035647.html,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Miami Heat,98.30,62,16,89.32,29,99.10,1.0,97.68,...,3.16,20130413035647.html,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Denver Nuggets,96.83,54,24,90.35,6,97.05,3.0,96.61,...,3.16,20130413035647.html,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,San Antonio Spurs,95.65,57,21,90.36,5,95.46,4.0,95.84,...,3.16,20130413035647.html,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Los Angeles Clippers,95.53,52,26,89.94,16,95.14,5.0,95.94,...,3.16,20130413035647.html,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--------------------------------------------------------------------------------


In [53]:
import os
import glob
import pandas as pd

# Define paths matching your layout
TARGET_DIR = "../data_curation/data"
OUTPUT_FILE = os.path.join(TARGET_DIR, "parquet_schema_audit.md")

def generate_schema_markdown(directory_path, output_md_path):
    """
    Scans the target directory for Parquet files and generates a structured
    Markdown schema inventory report saved directly to the target directory.
    """
    # Find all .parquet or .pq files in the directory recursively
    search_path = os.path.join(directory_path, "**", "*.parquet")
    parquet_files = glob.glob(search_path, recursive=True)
    
    search_path_pq = os.path.join(directory_path, "**", "*.pq")
    parquet_files.extend(glob.glob(search_path_pq, recursive=True))
    
    if not parquet_files:
        print(f"No Parquet files found in target directory: {directory_path}")
        return

    # Ensure we sort the files for a predictable, clean list structure
    parquet_files = sorted([f for f in parquet_files if os.path.isfile(f) and not f.endswith('.md')])

    md_content = []
    md_content.append("# Parquet Schema Inventory Report")
    md_content.append(f"**Generated Automatically:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    md_content.append(f"**Target Source Location:** `{directory_path}`\n")
    md_content.append("---")

    for file_path in parquet_files:
        # Get relative path from root folder for cleaner look
        relative_path = os.path.relpath(file_path, directory_path)
        md_content.append(f"\n## File: `{relative_path}`")
        
        try:
            # Read the dataset structure
            df = pd.read_parquet(file_path)
            
            total_rows = len(df)
            total_cols = len(df.columns)
            
            md_content.append(f"- **Total Record Count:** {total_rows:,}")
            md_content.append(f"- **Total Column Count:** {total_cols}\n")
            
            # Construct standard Markdown table headers
            md_content.append("| Index | Column Name | Data Type |")
            md_content.append("|---|---|---|")
            
            for idx, (col_name, dtype) in enumerate(df.dtypes.items(), start=1):
                # Escape potential markdown pipe characters embedded in column names if any
                clean_col_name = str(col_name).replace('|', '\|')
                md_content.append(f"| {idx} | `{clean_col_name}` | `{dtype}` |")
                
        except Exception as e:
            md_content.append(f"\n> ❌ **Error Reading Schema File:** {str(e)}\n")
        
        md_content.append("\n---")

    # Save the output file
    try:
        with open(output_md_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(md_content))
        print(f"Successfully generated schema file inventory at: {output_md_path}")
    except Exception as e:
        print(f"Failed to write markdown file: {e}")

# Run the schema summary generator script
generate_schema_markdown(TARGET_DIR, OUTPUT_FILE)

Successfully generated schema file inventory at: ../data_curation/data/parquet_schema_audit.md


In [58]:
df = pd.read_parquet('../data_curation/data/BoxScoresSummary.parquet')
df.head()

,game_id,game_code,game_status,game_status_text,period,game_time_utc,game_et,game_duration,attendance,sellout,...,home_score,away_team_id,away_team_tricode,away_team_name,away_team_city,away_q1,away_q2,away_q3,away_q4,away_score
0,0029600002,19961101/CLENJN,3,Final,4,1996-11-02T00:30:00Z,1996-11-01T19:30:00Z,2:10,18316,0,...,77,1610612739,CLE,Cavaliers,Cleveland,28,25,17,20,90
1,0029600001,19961101/CHIBOS,3,Final,4,1996-11-02T00:00:00Z,1996-11-01T19:00:00Z,2:15,18624,0,...,98,1610612741,CHI,Bulls,Chicago,27,21,32,27,107
2,0029600005,19961101/ATLMIA,3,Final,4,1996-11-02T00:30:00Z,1996-11-01T19:30:00Z,2:11,15113,0,...,94,1610612737,ATL,Hawks,Atlanta,15,24,27,15,81
3,0029600010,19961101/DALDEN,3,Final,4,1996-11-02T02:00:00Z,1996-11-01T21:00:00Z,2:15,16104,0,...,91,1610612742,DAL,Mavericks,Dallas,26,22,20,24,92
4,0029600009,19961101/SACHOU,3,Final,4,1996-11-02T01:30:00Z,1996-11-01T20:30:00Z,2:07,16285,0,...,96,1610612758,SAC,Kings,Sacramento,24,11,19,31,85
